# RAG Databricks Bluetab - Pipeline de Procesamiento de PDFs

Este notebook procesa archivos PDF de forma incremental, extrayendo texto y dividiéndolo en chunks para su posterior procesamiento.

## Características
- Procesamiento incremental
- Chunk size y overlap configurables
- Logging y monitoreo
- Tracking en MLflow

## Dependencias
- Ejecuta primero `00 Configuration and Utils`
- Ejecuta `01 Create needed tables` para asegurar la existencia de tablas

In [0]:
%run "./00 Configuration and Utils"


In [0]:
start_child_run("02_incremental_pdf_to_docs")

In [0]:
import mlflow

# Iniciar ejecución de MLflow para este paso
mlflow.log_param("step", "pdf_processing")
mlflow.log_param("environment", ENVIRONMENT)
mlflow.log_param("chunk_size", CHUNK_SIZE)
mlflow.log_param("chunk_overlap", CHUNK_OVERLAP)
mlflow.log_param("pdf_volume_path", PDF_VOLUME_PATH)

log_step("pdf_processing", "started", "Iniciando pipeline de procesamiento de PDFs")

## Instalar dependencias
Instala los paquetes Python necesarios para la extracción y procesamiento de PDFs.

In [0]:
# Instalar pdfplumber y langchain
%pip install pdfplumber langchain
dbutils.library.restartPython()

log_step("install_dependencies", "completed", "Dependencias instaladas correctamente")

In [0]:
%run "./00 Configuration and Utils"

## Escanear volumen de PDFs
Lista todos los archivos PDF en el volumen configurado.

In [0]:
import os
from pyspark.sql.functions import substring_index

log_step("scan_pdf_volume", "started", f"Escaneando {PDF_VOLUME_PATH}")

try:
    file_paths = [file.path for file in dbutils.fs.ls(PDF_VOLUME_PATH)]
    df = spark.createDataFrame(file_paths, "string").select(
        substring_index("value", "/", -1).alias("file_name")
    ).filter("file_name LIKE '%.pdf'")
    total_pdf_files = df.count()
    mlflow.log_metric("total_pdf_files_found", total_pdf_files)
    log_step("scan_pdf_volume", "success", f"Encontrados {total_pdf_files} archivos PDF")
    print(f"Archivos PDF encontrados en {PDF_VOLUME_PATH}:")
    df.show(truncate=False)
except Exception as e:
    log_step("scan_pdf_volume", "failed", f"Error escaneando volumen: {e}")
    raise e

## Identificar archivos nuevos para procesar
Compara los archivos encontrados con la tabla de tracking para procesar solo los nuevos.

In [0]:
log_step("identify_new_files", "started", "Buscando archivos no procesados")
try:
    processed_files_df = spark.sql(f"SELECT DISTINCT file_name FROM {DOCS_TRACK_TABLE_FULL}")
    processed_files = set(row["file_name"] for row in processed_files_df.collect())
    all_files = set(row["file_name"] for row in df.collect())
    new_files = list(all_files - processed_files)
    mlflow.log_metric("files_already_processed", len(processed_files))
    mlflow.log_metric("new_files_to_process", len(new_files))
    if new_files:
        log_step("identify_new_files", "success", f"Encontrados {len(new_files)} archivos nuevos para procesar")
        print("Archivos nuevos para procesar:")
        for file in new_files:
            print(f"  - {file}")
    else:
        log_step("identify_new_files", "success", "No hay archivos nuevos para procesar")
        print("Todos los archivos ya han sido procesados.")
except Exception as e:
    log_step("identify_new_files", "failed", f"Error identificando archivos nuevos: {e}")
    new_files = list(row["file_name"] for row in df.collect())
    log_step("identify_new_files", "fallback", f"Procesando todos los {len(new_files)} archivos")

## Extracción y chunking de texto
Extrae el texto de los PDFs nuevos y lo divide en chunks usando LangChain.

In [0]:
if new_files:
    log_step("text_extraction", "started", f"Procesando {len(new_files)} archivos")
    import pdfplumber
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    all_text = ''
    processed_files_info = []
    for file_name in new_files:
        try:
            log_step("extract_file", "started", f"Procesando {file_name}")
            pdf_path = os.path.join(PDF_VOLUME_PATH, file_name)
            file_text = ''
            page_count = 0
            with pdfplumber.open(pdf_path) as pdf:
                for pdf_page in pdf.pages:
                    single_page_text = pdf_page.extract_text()
                    if single_page_text:
                        file_text += f'\n{single_page_text}'
                        page_count += 1
            all_text += f'\n\n--- FILE: {file_name} ---\n{file_text}'
            file_size = len(file_text)
            processed_files_info.append({
                'file_name': file_name,
                'page_count': page_count,
                'text_length': file_size
            })
            log_step("extract_file", "success", f"{file_name}: {page_count} páginas, {file_size} caracteres")
        except Exception as e:
            log_step("extract_file", "failed", f"Error procesando {file_name}: {e}")
            continue
    total_text_length = len(all_text)
    mlflow.log_metric("total_text_extracted_chars", total_text_length)
    mlflow.log_metric("files_successfully_processed", len(processed_files_info))
    log_step("text_extraction", "completed", f"Extraídos {total_text_length} caracteres")
else:
    all_text = ''
    processed_files_info = []

if new_files and all_text.strip():
    log_step("text_chunking", "started", f"Dividiendo texto con chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", " ", ""],
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
    )
    chunks = splitter.split_text(all_text)
    mlflow.log_metric("total_chunks_created", len(chunks))
    mlflow.log_metric("avg_chunk_length", sum(len(chunk) for chunk in chunks) / len(chunks) if chunks else 0)
    log_step("text_chunking", "success", f"Creados {len(chunks)} chunks")
    if chunks:
        print("Ejemplo de chunk (primeros 200 caracteres):")
        print(chunks[0][:200] + "..." if len(chunks[0]) > 200 else chunks[0])
else:
    log_step("text_chunking", "skipped", "No hay texto nuevo para procesar")
    chunks = []

## Insertar chunks en la tabla docs_text
Almacena los chunks procesados en la tabla docs_text.

In [0]:
if chunks:
    log_step("udf_creation", "started", "Creando UDF para chunks")
    from pyspark.sql.functions import pandas_udf
    from pyspark.sql.types import ArrayType, StringType
    import pandas as pd
    @pandas_udf("array<string>")
    def get_chunks(dummy):
        return pd.Series([chunks])
    spark.udf.register("get_chunks_udf", get_chunks)
    log_step("udf_creation", "success", "UDF creada y registrada")

    log_step("data_insertion", "started", f"Insertando {len(chunks)} chunks en {DOCS_TEXT_TABLE_FULL}")
    try:
        spark.sql(f"""
            INSERT INTO {DOCS_TEXT_TABLE_FULL} (text)
            SELECT explode(get_chunks_udf('dummy')) as text
        """)
        total_records = spark.sql(f"SELECT COUNT(*) as count FROM {DOCS_TEXT_TABLE_FULL}").collect()[0]["count"]
        mlflow.log_metric("total_records_after_insertion", total_records)
        log_step("data_insertion", "success", f"Chunks insertados correctamente. Total registros: {total_records}")
    except Exception as e:
        log_step("data_insertion", "failed", f"Error insertando chunks: {e}")
        raise e
else:
    log_step("data_insertion", "skipped", "No hay chunks para insertar")

## Actualizar tabla de tracking
Registra los archivos procesados en la tabla docs_track para evitar reprocesamiento.

In [0]:
if new_files:
    log_step("update_tracking", "started", f"Actualizando tabla de tracking para {len(new_files)} archivos")
    try:
        df.createOrReplaceTempView("temp_new_files")
        spark.sql(f"""
            INSERT INTO {DOCS_TRACK_TABLE_FULL} (file_name)
            SELECT file_name FROM temp_new_files
            WHERE NOT EXISTS (
                SELECT 1 FROM {DOCS_TRACK_TABLE_FULL} track
                WHERE temp_new_files.file_name = track.file_name
            )
        """)
        tracked_files_count = spark.sql(f"SELECT COUNT(*) as count FROM {DOCS_TRACK_TABLE_FULL}").collect()[0]["count"]
        mlflow.log_metric("total_tracked_files", tracked_files_count)
        log_step("update_tracking", "success", f"Tabla de tracking actualizada. Total archivos: {tracked_files_count}")
    except Exception as e:
        log_step("update_tracking", "failed", f"Error actualizando tabla de tracking: {e}")
        raise e
else:
    log_step("update_tracking", "skipped", "No hay archivos nuevos para registrar")

## Resumen final y limpieza
Muestra un resumen de los resultados del procesamiento y registra métricas finales en MLflow.

In [0]:
try:
    total_text_records = spark.sql(f"SELECT COUNT(*) as count FROM {DOCS_TEXT_TABLE_FULL}").collect()[0]["count"]
    total_tracked_files = spark.sql(f"SELECT COUNT(*) as count FROM {DOCS_TRACK_TABLE_FULL}").collect()[0]["count"]
    mlflow.log_metric("final_text_records", total_text_records)
    mlflow.log_metric("final_tracked_files", total_tracked_files)
    summary = {
        "files_processed": len(new_files) if new_files else 0,
        "chunks_created": len(chunks) if chunks else 0,
        "total_text_records": total_text_records,
        "total_tracked_files": total_tracked_files
    }
    mlflow.log_dict(summary, "processing_summary.json")
    log_step("pdf_processing", "completed", f"Procesamiento completo: {summary}")
    print("="*60)
    print("RESUMEN DE PROCESAMIENTO DE PDF")
    print("="*60)
    for key, value in summary.items():
        print(f"{key.replace('_', ' ').title()}: {value}")
    print("="*60)
except Exception as e:
    log_step("pdf_processing", "failed", f"Error en resumen final: {e}")
    raise e

In [0]:
# Finalizar child run
try:
    log_step("table_creation", "completed", "Proceso de creación de tablas finalizado")
    end_child_run("success")
    print("✅ Child run finalizada correctamente")
except Exception as e:
    print(f"⚠️ Error finalizando child run: {e}")
    end_child_run("failed")